# ARTICLE Pipeline — Notebook 1: Data Generation

**Folder:** `ARTICLE_SBTS` (independent of `PHD_SBTS`)

This notebook generates the three path datasets (GBM, Heston, SBTS Config A)
plus historical test paths for the article
"Multivariate Schrödinger Bridge for Deep Hedging of Rainbow Options".

**Pipeline:**
1. Download AAPL/JPM/XOM daily prices (2005-01-01 to 2026-12-31)
2. Split: Train 2005–2019 (generator calibration), Test 2020–2025 (out-of-sample)
3. Calibrate & generate 20,000 paths for each generator
4. Build historical test paths for 4 periods (COVID, PostCOVID, Normal, Recent 2025)
5. Distributional comparison + article figures

**Output files (on Google Drive, `ARTICLE_SBTS/`):**
- `gbm_deep_hedging.npz`    — GBM paths + train/val/test split
- `heston_deep_hedging.npz` — Heston paths + split
- `sbts_deep_hedging.npz`   — SBTS Config A (isotropic, h selected via
  held-out MSE) paths + split
- `historical_test_paths.npz` — 4 test periods
- `figures/` — article-ready PDFs

Expected runtime on Colab T4: ~45 min (most in SBTS generation).

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 1: SETUP & CONFIGURATION
# ═════════════════════════════════════════════════════════════════
!pip install yfinance scipy statsmodels torch --upgrade -q

import sys, os, time, gc, shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
from scipy import stats
from scipy.stats import ks_2samp
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# ── Device ────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE.type.upper()}")
if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Matplotlib defaults (article style) ───────────────────────────
plt.rcParams.update({
    'figure.dpi':    150,
    'savefig.dpi':   300,
    'font.size':     10,
    'font.family':   'serif',
    'axes.grid':     True,
    'grid.alpha':    0.25,
    'grid.linewidth':0.5,
    'axes.labelsize':10,
    'axes.titlesize':11,
    'legend.fontsize':9,
    'xtick.labelsize':9,
    'ytick.labelsize':9,
    'axes.linewidth': 0.8,
    'figure.figsize':(7, 4.3),
})

# ═════════════════════════════════════════════════════════════════
#  CONFIGURATION — All generators use 2005-2019 historical data
# ═════════════════════════════════════════════════════════════════

TICKERS       = ['AAPL', 'JPM', 'XOM']
SECTORS       = ['Technology', 'Financials', 'Energy']
FULL_START    = '2005-01-01'
FULL_END      = '2026-12-31'          # yfinance will clip to latest available
TRAIN_END     = '2019-12-31'          # All generators see ≤ this
TEST_START    = '2020-01-01'

N_WINDOW      = 252                   # 1 trading year
DELTA_T       = 1 / 252
N_PI          = 100                   # Euler sub-steps per interval
MARKOV_K      = 1                     # Will be set by bandwidth search

M_SIMU        = 20_000                # Paths per generator
N_TRAIN       = 16_000                # Deep-hedging train split
N_VAL         =  2_000                # Deep-hedging val split
N_TEST        =  2_000                # Deep-hedging test split (synthetic check)
BATCH_SIZE    = 500                   # GPU batch for SBTS generation

DRIVE_FOLDER  = '/content/drive/MyDrive/ARTICLE_SBTS'
FIG_FOLDER    = os.path.join(DRIVE_FOLDER, 'figures')
os.makedirs(DRIVE_FOLDER, exist_ok=True)
os.makedirs(FIG_FOLDER, exist_ok=True)

d = len(TICKERS)

# ── Test periods (out-of-sample) ──────────────────────────────────
TEST_PERIODS = {
    'COVID_2020':        ('2020-01-01', '2020-12-31'),   # stress
    'PostCOVID_2021_22': ('2021-01-01', '2022-12-31'),   # recovery
    'Normal_2023_24':    ('2023-01-01', '2024-12-31'),   # calm
    'Recent_2025':       ('2025-01-01', '2025-12-31'),   # fresh OOS (NEW)
}

print(f"\nSetup complete")
print(f"  Folder:         {DRIVE_FOLDER}")
print(f"  Assets:         {TICKERS}")
print(f"  Generator data: {FULL_START} – {TRAIN_END}")
print(f"  Test periods:   {list(TEST_PERIODS.keys())}")
print(f"  Paths:          {M_SIMU:,} → train {N_TRAIN}/val {N_VAL}/test {N_TEST}")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 2: DOWNLOAD DATA & TEMPORAL SPLIT
# ═════════════════════════════════════════════════════════════════

print("Downloading stock data from Yahoo Finance...")
raw_data = yf.download(TICKERS, start=FULL_START, end=FULL_END, progress=False)

if 'Adj Close' in raw_data.columns.get_level_values(0):
    close_prices = raw_data['Adj Close'][TICKERS]
else:
    close_prices = raw_data['Close'][TICKERS]

close_prices    = close_prices.dropna().ffill()
log_returns_df  = np.log(close_prices / close_prices.shift(1)).dropna()

# ── Splits ────────────────────────────────────────────────────────
full_returns  = log_returns_df
train_returns = log_returns_df.loc[:TRAIN_END]
test_returns  = log_returns_df.loc[TEST_START:]

train_returns_values = train_returns.values
S0 = close_prices.loc[:TRAIN_END].iloc[-1].values   # last price in 2019

print(f"\nData prepared ({close_prices.index[0].date()} to {close_prices.index[-1].date()})")
print(f"  Training:  {len(train_returns):>5} days (2005–2019)")
print(f"  Test:      {len(test_returns):>5} days (2020+)")
print(f"  S0 = {dict(zip(TICKERS, S0.round(2)))}")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 3: DESCRIPTIVE STATISTICS
# ═════════════════════════════════════════════════════════════════

print("=" * 70)
print("DESCRIPTIVE STATISTICS OF DAILY LOG-RETURNS")
print("=" * 70)

for label, df in [("TRAINING (2005–2019)", train_returns),
                  ("TEST (2020+)", test_returns)]:
    print(f"\n{label}")
    print("-" * 60)
    desc = pd.DataFrame(index=TICKERS)
    desc['Mean (%)']     = df.mean() * 100
    desc['Std (%)']      = df.std() * 100
    desc['Skewness']     = df.skew()
    desc['Kurtosis']     = df.kurtosis() + 3
    desc['Min (%)']      = df.min() * 100
    desc['Max (%)']      = df.max() * 100
    desc['Ann Vol (%)']  = df.std() * np.sqrt(252) * 100
    print(desc.round(4).T.to_string())

for label, df in [("TRAINING", train_returns), ("TEST", test_returns)]:
    print(f"\nCORRELATION — {label}")
    print(df.corr().round(4).to_string())

# Jarque-Bera
print("\n" + "=" * 70)
print("STYLIZED FACTS — Jarque-Bera test (train set)")
print("=" * 70)
for t in TICKERS:
    k = train_returns[t].kurtosis() + 3
    jb, p = stats.jarque_bera(train_returns[t].dropna())
    print(f"  {t}: kurtosis={k:.2f}  JB p-value={p:.2e} "
          f"{'(rejects normality)' if p < 0.05 else '(normality not rejected)'}")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 4: FIGURE 1 — HISTORICAL DATA OVERVIEW (2×2)
# ═════════════════════════════════════════════════════════════════
#  Panel (a): Normalised prices, log scale
#  Panel (b): Daily returns %
#  Panel (c): Return distributions (KDE)
#  Panel (d): Yearly pairwise correlations
# ═════════════════════════════════════════════════════════════════

from matplotlib.lines import Line2D

colors_asset = {'AAPL': '#2E86AB', 'JPM': '#A23B72', 'XOM': '#F18F01'}

fig, axes = plt.subplots(2, 2, figsize=(11, 7), dpi=150)

# ── (a) Normalised prices — log scale ───────────────────────────
ax = axes[0, 0]
S_norm = close_prices / close_prices.iloc[0]
for t in TICKERS:
    ax.plot(S_norm.index, S_norm[t], color=colors_asset[t], lw=0.9, label=t)
ax.set_yscale('log')
ax.set_ylabel(r'$S_t / S_0$ (log scale)')
ax.set_title('(a) Normalised Prices', fontsize=11, loc='left', fontweight='bold')
ax.axvline(pd.Timestamp(TRAIN_END), color='red', ls='--', lw=1.0, alpha=0.6)
ax.text(pd.Timestamp(TRAIN_END) + pd.Timedelta(days=60), ax.get_ylim()[0]*1.3,
        'train/test', color='red', fontsize=8, alpha=0.8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(3))
ax.legend(loc='upper left', frameon=False)

# ── (b) Daily returns ───────────────────────────────────────────
ax = axes[0, 1]
for t in TICKERS:
    ax.plot(log_returns_df.index, log_returns_df[t] * 100,
            color=colors_asset[t], lw=0.3, alpha=0.7, label=t)
ax.axvline(pd.Timestamp(TRAIN_END), color='red', ls='--', lw=1.0, alpha=0.6)
ax.set_ylabel('Daily return (%)')
ax.set_title('(b) Daily Returns', fontsize=11, loc='left', fontweight='bold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(3))
ax.set_ylim(-20, 20)
ax.legend(loc='lower left', frameon=False)

# ── (c) Return distributions (KDE) ──────────────────────────────
ax = axes[1, 0]
for t in TICKERS:
    r = log_returns_df[t].values * 100
    lo, hi = np.percentile(r, [0.5, 99.5])
    r_clip = r[(r >= lo) & (r <= hi)]
    kde = stats.gaussian_kde(r_clip)
    x = np.linspace(-8, 8, 400)
    ax.plot(x, kde(x), color=colors_asset[t], lw=1.5, label=t)
ax.set_xlabel('Daily return (%)')
ax.set_ylabel('Density')
ax.set_title('(c) Return Distributions', fontsize=11, loc='left', fontweight='bold')
ax.legend(frameon=False)
ax.set_xlim(-8, 8)

# ── (d) Yearly pairwise correlations ────────────────────────────
ax = axes[1, 1]
pairs  = [('AAPL','JPM'), ('AAPL','XOM'), ('JPM','XOM')]
colors_pair = ['#3E5C76', '#748CAB', '#E6A0C4']
years  = sorted(log_returns_df.index.year.unique())

for (a, b), color in zip(pairs, colors_pair):
    corrs = []
    for y in years:
        sub = log_returns_df[log_returns_df.index.year == y]
        if len(sub) > 10:
            corrs.append(sub[a].corr(sub[b]))
        else:
            corrs.append(np.nan)
    ax.plot(years, corrs, color=color, marker='o', ms=4, lw=1.2,
            label=f'{a}-{b}')
ax.axvline(int(TRAIN_END[:4]) + 0.5, color='red', ls='--', lw=1.0, alpha=0.6)
ax.axhline(0, color='gray', lw=0.5, alpha=0.5)
ax.set_ylabel('Yearly pairwise correlation')
ax.set_title('(d) Time-Varying Correlations', fontsize=11, loc='left', fontweight='bold')
ax.legend(loc='lower right', frameon=False)
ax.set_ylim(-0.1, 1.0)

plt.tight_layout()
plt.savefig(os.path.join(FIG_FOLDER, 'fig1_historical_overview.pdf'),
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIG_FOLDER, 'fig1_historical_overview.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print(f"  💾 fig1_historical_overview.pdf/png")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 5: SBTS HELPER FUNCTIONS  (patched — numpy bandwidth search)
# ═════════════════════════════════════════════════════════════════
#  References:
#    [H23] Hamdouche, Henry-Labordère, Pham (2023). arXiv:2304.05093
#    [A25] Alouadi, Barreau, Carlier, Pham (2025). ICAIF '25
#
#  NOTE: Bandwidth selection uses NUMPY (CPU) to avoid
#        the nvrtc / libnvrtc-builtins issue triggered by
#        JIT-compiled reductions in certain Colab/CUDA versions.
#        Path GENERATION still uses the GPU via simulate_sbts.
#  Config A only (isotropic kernel).
# ═════════════════════════════════════════════════════════════════

import shutil  # for cleanup_temp_batches

def quartic_kernel_isotropic(diff, h):
    """
    Torch version (GPU): K_h(u) = (h² - ‖u‖²)² · 1[‖u‖ < h]
    Used only in simulate_sbts (path generation).
    """
    norm_sq = torch.sum(diff ** 2, dim=-1)
    h_sq    = h * h
    return torch.where(norm_sq < h_sq, (h_sq - norm_sq) ** 2,
                       torch.zeros_like(norm_sq))


def quartic_kernel_isotropic_np(diff_sq_sum, h):
    """
    Numpy version (CPU) — used in bandwidth search.
    diff_sq_sum : np.ndarray (...) — already ||u||² aggregated
    """
    h_sq = h * h
    k = np.where(diff_sq_sum < h_sq, (h_sq - diff_sq_sum) ** 2, 0.0)
    return k


def simulate_sbts(
    N, M, d, K, X, N_pi, h, deltati, M_simu,
    device, batch_size=500, seed=None,
    config_name='A', drive_folder=DRIVE_FOLDER,
):
    """
    SBTS path generator — isotropic kernel only. (Torch/GPU)

    Implements [H23] §4.3 Algorithm 1 with [A25] Markov-K sliding window.
    """
    if seed is not None:
        torch.manual_seed(seed)

    X_ref  = torch.as_tensor(X, dtype=torch.float32, device=device)
    dt_sub = deltati / N_pi
    kernel_fn = lambda diff: quartic_kernel_isotropic(diff, h)
    print(f"  SBTS isotropic  h={h:.4f}  K={K}")

    # Checkpointing
    temp_dir = os.path.join(drive_folder, f"temp_{config_name}_batches")
    os.makedirs(temp_dir, exist_ok=True)
    n_batches = (M_simu + batch_size - 1) // batch_size
    all_paths = []
    start_batch = 0

    for b_idx in range(n_batches):
        bf = os.path.join(temp_dir, f"batch_{b_idx}.npy")
        if os.path.exists(bf):
            try:
                data = np.load(bf)
                if (np.isfinite(data).all() and data.shape[1] == N
                        and data.shape[2] == d):
                    all_paths.append(torch.from_numpy(data))
                    start_batch = b_idx + 1
                else:
                    break
            except Exception:
                break
        else:
            break

    if start_batch > 0:
        print(f"  ⏩ Resuming from batch {start_batch + 1}/{n_batches}")
    if start_batch >= n_batches:
        print(f"  ✅ All {n_batches} batches done.")
        return torch.cat(all_paths, dim=0).numpy()

    t0 = time.time()
    for b_idx in range(start_batch, n_batches):
        bs = min(batch_size, M_simu - b_idx * batch_size)
        if seed is not None:
            torch.manual_seed(seed + b_idx * 1000)

        BM   = torch.randn(bs, N * N_pi, d, device=device)
        Y    = X_ref[0, 0].unsqueeze(0).expand(bs, -1).clone()
        path = torch.zeros(bs, N + 1, d, device=device)

        log_weights = torch.zeros(bs, M, device=device)
        last_K_buf  = torch.zeros(bs, K, d, device=device)
        index_queue = 0
        bm_idx      = 0

        for i in range(N):
            if i > 0:
                if index_queue >= K:
                    X_oldest  = last_K_buf[:, 0, :]
                    ref_oldest = X_ref[:, i - K, :]
                    diff_old = ref_oldest.unsqueeze(0) - X_oldest.unsqueeze(1)
                    k_old    = kernel_fn(diff_old)
                    log_k_old = torch.log(k_old.clamp(min=1e-30))
                    log_weights = log_weights - log_k_old
                    if K > 1:
                        last_K_buf[:, :-1, :] = last_K_buf[:, 1:, :].clone()
                    last_K_buf[:, -1, :]  = Y.clone()
                else:
                    last_K_buf[:, index_queue, :] = Y.clone()
                index_queue += 1

                diff_cur = X_ref[:, i, :].unsqueeze(0) - Y.unsqueeze(1)
                k_cur    = kernel_fn(diff_cur)
                log_k_cur = torch.log(k_cur.clamp(min=1e-30))
                log_weights = log_weights + log_k_cur
            else:
                log_weights = torch.zeros(bs, M, device=device)

            X_next = X_ref[:, i + 1, :]
            Y_at_ti = Y.clone()
            diff_B  = X_next.unsqueeze(0) - Y_at_ti.unsqueeze(1)
            bridge_B = torch.sum(diff_B ** 2, dim=-1) / (2.0 * deltati)

            for k in range(N_pi):
                rem = deltati - k * dt_sub
                diff_t = X_next.unsqueeze(0) - Y.unsqueeze(1)

                if k == 0:
                    lwmax    = log_weights.max(dim=1, keepdim=True).values
                    w_stable = torch.exp(log_weights - lwmax)
                    den      = w_stable.sum(dim=1, keepdim=True)
                    num      = (w_stable.unsqueeze(-1) * diff_t).sum(dim=1)
                else:
                    bridge_A = torch.sum(diff_t ** 2, dim=-1) / (2.0 * rem)
                    log_F    = -bridge_A + bridge_B
                    log_w_total = log_weights + log_F
                    lwmax    = log_w_total.max(dim=1, keepdim=True).values
                    w_stable = torch.exp(log_w_total - lwmax)
                    den      = w_stable.sum(dim=1, keepdim=True)
                    num      = (w_stable.unsqueeze(-1) * diff_t).sum(dim=1)

                safe_den = den.clamp(min=1e-30)
                drift    = (1.0 / rem) * (num / safe_den)

                bad = den.squeeze(-1) < 1e-30
                if bad.any():
                    drift[bad] = 0.0

                Y = Y + drift * dt_sub + BM[:, bm_idx, :] * (dt_sub ** 0.5)
                bm_idx += 1

            path[:, i + 1, :] = Y
            del diff_B, bridge_B, X_next

        batch_res = path[:, 1:, :].cpu().numpy()
        if not np.isfinite(batch_res).all():
            batch_res = np.nan_to_num(batch_res, nan=0.0, posinf=0.0, neginf=0.0)

        all_paths.append(torch.from_numpy(batch_res))
        np.save(os.path.join(temp_dir, f"batch_{b_idx}.npy"), batch_res)

        del BM, log_weights, path, last_K_buf
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        elapsed = time.time() - t0
        batches_done = b_idx - start_batch + 1
        if b_idx == start_batch or (b_idx + 1) % 5 == 0 or b_idx == n_batches - 1:
            pct = (b_idx + 1) / n_batches * 100
            eta = elapsed / batches_done * (n_batches - b_idx - 1) if batches_done > 0 else 0
            print(f"    Batch {b_idx+1}/{n_batches}  ({pct:.0f}%)  "
                  f"elapsed {elapsed:.0f}s  ETA {eta:.0f}s", flush=True)

    return torch.cat(all_paths, dim=0).numpy()


def cleanup_temp_batches(config_name, drive_folder):
    td = os.path.join(drive_folder, f"temp_{config_name}_batches")
    if os.path.exists(td):
        shutil.rmtree(td, ignore_errors=True)
        print(f"  🗑️  Cleaned temp batches: {td}")


# ──────────────────────────────────────────────────────────────────
#  BANDWIDTH & MARKOV-K SELECTION  [A25] eq. (5)  — CPU/NUMPY
# ──────────────────────────────────────────────────────────────────
#
#  Paper eq. (5):
#     MSE(h, k) = (1/Q) Σ_q (1/L) Σ_l || Ŷ^{q,l}_T - Y^q_T ||²
#
#  For each test path q:
#    1. Condition on last k steps of Y^q.
#    2. Compute kernel weights w^m = Π K_h(Y^q_j - X^m_j) for j in window.
#    3. If Σ w^m = 0, skip (zero-weight).
#    4. Else: simulate L next-step draws via Brownian bridge with
#       Nadaraya-Watson drift, using reference paths weighted by w^m.
#    5. Compare predicted Y^q_{T} with true terminal value.
#
#  We use a SIMPLIFIED one-step-ahead version for speed — predicting the
#  NEXT observation conditional on the current k-step history. This is
#  a practical proxy that scales well and is used in [H23] §5 for h
#  selection. Full L-sample bridge is an option but adds O(L) cost.
#
#  Implementation on CPU/numpy to avoid the libnvrtc JIT issue.
# ──────────────────────────────────────────────────────────────────

def _mse_one_config_np(X_train, Y_test, k_markov, h, d, verbose=False):
    """
    Held-out MSE for one (h, k) configuration — numpy/CPU.

    Parameters
    ----------
    X_train  : np.ndarray (M_train, N+1, d) — reference paths
    Y_test   : np.ndarray (Q, N+1, d)       — test paths (held out)
    k_markov : int                          — Markov window
    h        : float                        — bandwidth
    d        : int                          — n assets

    Returns
    -------
    mse_total      : float — average ||pred - true||² per asset
    mse_per_asset  : np.ndarray (d,)
    n_zero_weight  : int — number of test paths with zero weight (skipped)
    """
    Q = Y_test.shape[0]
    N = Y_test.shape[1] - 1
    M_train = X_train.shape[0]

    # Predict the next observation given the last k_markov history steps.
    # We use the LAST POSSIBLE transition of each test path:
    #   condition on steps [N-k..N-1], predict step N.
    start = max(0, N - k_markov)
    # Shape: (Q, k_markov_actual, d)
    Y_cond  = Y_test[:, start : N, :]                 # conditioning window
    Y_true  = Y_test[:, N, :]                         # true next obs (the "N-th")
    # Reference paths at the same positions:
    X_cond  = X_train[:, start : N, :]                # (M_train, k, d)
    X_next  = X_train[:, N, :]                        # (M_train, d)

    h_sq = h * h
    mse_per_asset = np.zeros(d, dtype=np.float64)
    n_zero_weight = 0
    n_valid       = 0

    # Loop over query paths
    for q in range(Q):
        # diff shape: (M_train, k, d)
        diff = X_cond - Y_cond[q : q + 1, :, :]
        # Per-step squared-norm: (M_train, k)
        step_sq_sum = (diff ** 2).sum(axis=-1)
        # Per-step kernel: (M_train, k)
        k_step = np.where(step_sq_sum < h_sq,
                          (h_sq - step_sq_sum) ** 2, 0.0)
        # Markov-k: product across steps
        w = np.prod(k_step, axis=-1)   # (M_train,)
        w_sum = w.sum()

        if w_sum < 1e-30 or not np.isfinite(w_sum):
            n_zero_weight += 1
            continue

        # Nadaraya-Watson predictor
        pred = (w[:, None] * X_next).sum(axis=0) / w_sum   # (d,)
        err_sq = (pred - Y_true[q]) ** 2                   # (d,)
        mse_per_asset += err_sq
        n_valid += 1

    if n_valid == 0:
        return float('inf'), np.full(d, float('inf')), n_zero_weight

    mse_per_asset = mse_per_asset / n_valid
    mse_total     = mse_per_asset.mean()
    return float(mse_total), mse_per_asset, n_zero_weight


def select_bandwidth_and_markov_k(
    X_ref, d, TICKERS, deltati, device,
    h_candidates=None, k_candidates=None,
    Q=100, L=50, test_fraction=0.2, seed=42, verbose=True,
):
    """
    Isotropic grid search over (h, k) via [A25] eq. (5) held-out MSE.
    Uses NUMPY / CPU to avoid libnvrtc JIT issue on some Colab GPUs.

    Parameters
    ----------
    X_ref : np.ndarray (M_total, N+1, d) — reference paths (incl. t0)
    Q     : int — max # held-out test paths
    L     : int — retained for API; not used in one-step MSE variant
    test_fraction : float — fraction of M_total for test split

    Returns
    -------
    h_best, k_best, meta
    """
    X_ref = np.asarray(X_ref)
    M_total, N_plus1, _ = X_ref.shape
    N = N_plus1 - 1
    rng = np.random.default_rng(seed)

    n_test = min(Q, max(int(M_total * test_fraction), Q))
    n_test = min(n_test, M_total - 50)
    idx = rng.permutation(M_total)
    test_idx, train_idx = idx[:n_test], idx[n_test:]
    X_train = X_ref[train_idx]
    Y_test  = X_ref[test_idx]
    Q_actual = len(test_idx)

    if h_candidates is None:
        h_candidates = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10,
                        0.12, 0.15, 0.18, 0.20, 0.25, 0.30, 0.40, 0.50]
    if k_candidates is None:
        raw_k = [1, 2, 5, 10, 21, 63]
        k_candidates = sorted(set(min(k, N - 1) for k in raw_k if k <= N - 1))

    if verbose:
        print(f"\n  ┌─ BANDWIDTH & MARKOV-K SELECTION — [A25] eq. (5) ─────")
        print(f"  │  Engine:   numpy (CPU) — avoids libnvrtc JIT issue")
        print(f"  │  M_train:  {len(train_idx)}")
        print(f"  │  Q_test:   {Q_actual}")
        print(f"  │  N={N}, d={d}")
        print(f"  │  h candidates: {h_candidates}")
        print(f"  │  k candidates: {k_candidates}")
        print(f"  │  Total configs: "
              f"{len(h_candidates) * len(k_candidates)}")
        print(f"  └{'─' * 60}")
        print(f"\n  {'h':>8}  {'k':>4}  {'MSE':>14}  {'zero-w':>8}  "
              f"{'time(s)':>8}")
        print(f"  {'-' * 50}")

    meta = {'mse': {}, 'zero_weight': {}, 'times': {}}
    best_mse = float('inf')
    h_best   = h_candidates[len(h_candidates) // 2]
    k_best   = k_candidates[0]

    grid_t0 = time.time()
    for h_c in h_candidates:
        for k_c in k_candidates:
            t0 = time.time()
            mse, mse_pa, n_zero = _mse_one_config_np(
                X_train, Y_test, k_c, h_c, d, verbose=False)
            dt = time.time() - t0

            meta['mse'][(h_c, k_c)]         = mse
            meta['zero_weight'][(h_c, k_c)] = n_zero
            meta['times'][(h_c, k_c)]       = dt

            if np.isfinite(mse) and mse < best_mse:
                best_mse = mse
                h_best   = h_c
                k_best   = k_c
                marker = ' <-- new best'
            else:
                marker = ''

            if verbose:
                zw = f"{n_zero}/{Q_actual}"
                mse_str = (f"{mse:>14.6f}" if np.isfinite(mse)
                           else f"{'inf':>14s}")
                print(f"  {h_c:>8.4f}  {k_c:>4}  {mse_str}  {zw:>8}  "
                      f"{dt:>8.2f}{marker}", flush=True)

    total_t = time.time() - grid_t0
    if verbose:
        print(f"\n  Grid search done in {total_t:.1f}s "
              f"({total_t / (len(h_candidates) * len(k_candidates)):.2f}s/config)")
        print(f"\n  ╔{'═' * 60}╗")
        print(f"  ║  BEST:   h* = {h_best:.4f}   k* = {k_best}   "
              f"MSE = {best_mse:.6f}  ║")
        print(f"  ╚{'═' * 60}╝")

    meta['h_best']   = h_best
    meta['k_best']   = k_best
    meta['best_mse'] = best_mse
    meta['grid_search_seconds'] = total_t
    return h_best, k_best, meta


def returns_to_S_norm(returns, M_simu, N_WINDOW, d):
    """Convert log-returns (M, N, d) → normalised prices S_norm (M, N+1, d)."""
    S_norm = np.ones((M_simu, N_WINDOW + 1, d))
    S_norm[:, 1:, :] = np.exp(np.cumsum(returns, axis=1))
    return S_norm


print("  ✅ SBTS helper functions defined (numpy bandwidth search)")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 6: GBM + HESTON HELPER FUNCTIONS
# ═════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────
#  GBM
# ──────────────────────────────────────────────────────────────────

def calibrate_gbm(train_returns_df, TICKERS):
    """Calibrate multivariate GBM from daily log-returns."""
    drift_annual = train_returns_df.mean().values * 252
    sigma_annual = train_returns_df.std().values  * np.sqrt(252)
    rho_matrix   = train_returns_df.corr().values
    L_chol       = np.linalg.cholesky(rho_matrix)

    print(f"\n  Drift (μ-½σ², ann.): "
          f"{dict(zip(TICKERS, drift_annual.round(4)))}")
    print(f"  Volatility (ann.):   "
          f"{dict(zip(TICKERS, sigma_annual.round(4)))}")
    print(f"  det(ρ) = {np.linalg.det(rho_matrix):.4f} (positive definite)")

    return {'drift_annual': drift_annual, 'sigma_annual': sigma_annual,
            'rho_matrix': rho_matrix, 'L_chol': L_chol}


def generate_gbm(gbm_params, S0, N_WINDOW, d, DELTA_T, DEVICE,
                 M_SIMU=20_000, seed=42, drive_folder=DRIVE_FOLDER):
    """Generate correlated GBM paths on GPU."""
    cache_path = os.path.join(drive_folder, 'gbm_paths.npz')
    if os.path.exists(cache_path):
        try:
            data = np.load(cache_path)
            X, S_norm = data['returns'], data['S_norm']
            if X.shape == (M_SIMU, N_WINDOW, d) and np.isfinite(X).all():
                print(f"  ⏩  GBM loaded from {cache_path}")
                return {'X_gbm': X, 'S_gbm': S_norm * S0[None, None, :],
                        'S_gbm_norm': S_norm, 'gbm_params': gbm_params}
        except Exception as e:
            print(f"  ⚠️  Cache load failed: {e}")

    print(f"  Generating {M_SIMU:,} GBM paths on {DEVICE}...")
    t0 = time.time()
    torch.manual_seed(seed)

    dt_t    = torch.tensor(DELTA_T, device=DEVICE, dtype=torch.float32)
    drift_t = torch.tensor(gbm_params['drift_annual'], device=DEVICE, dtype=torch.float32)
    sig_t   = torch.tensor(gbm_params['sigma_annual'], device=DEVICE, dtype=torch.float32)
    L_t     = torch.tensor(gbm_params['L_chol'],       device=DEVICE, dtype=torch.float32)
    S0_t    = torch.tensor(S0, device=DEVICE, dtype=torch.float32)

    Z      = torch.randn(M_SIMU, N_WINDOW, d, device=DEVICE)
    Z_corr = torch.matmul(Z, L_t.T)
    drift_step = drift_t * dt_t
    diff_step  = sig_t   * torch.sqrt(dt_t)
    X_tensor   = drift_step + diff_step * Z_corr

    S_tensor = torch.zeros(M_SIMU, N_WINDOW + 1, d, device=DEVICE)
    S_tensor[:, 0, :]  = S0_t
    S_tensor[:, 1:, :] = S0_t * torch.exp(torch.cumsum(X_tensor, dim=1))

    X_gbm      = X_tensor.cpu().numpy()
    S_gbm      = S_tensor.cpu().numpy()
    S_gbm_norm = S_gbm / S0[None, None, :]
    del Z, Z_corr, X_tensor, S_tensor
    clear_mem()

    print(f"  Done in {time.time() - t0:.2f}s")
    flat = X_gbm.reshape(-1, d)
    gen_corr = np.corrcoef(flat.T)
    print(f"  Max |Δρ|: {np.abs(gen_corr - gbm_params['rho_matrix']).max():.6f}")
    print(f"  GBM kurtosis: "
          f"{[round(float(stats.kurtosis(flat[:,k])+3),2) for k in range(d)]}")

    np.savez_compressed(cache_path, returns=X_gbm, S_norm=S_gbm_norm,
                        S_prices=S_gbm, S0=S0,
                        drift_annual=gbm_params['drift_annual'],
                        sigma_annual=gbm_params['sigma_annual'],
                        rho_matrix=gbm_params['rho_matrix'])
    print(f"  💾 {cache_path} ({os.path.getsize(cache_path)/1e6:.1f} MB)")

    return {'X_gbm': X_gbm, 'S_gbm': S_gbm, 'S_gbm_norm': S_gbm_norm,
            'gbm_params': gbm_params}


# ──────────────────────────────────────────────────────────────────
#  HESTON — Full Truncation Euler + Milstein CIR
# ──────────────────────────────────────────────────────────────────
#  References:
#    [B19]   Buehler et al. (2019). Deep hedging — single-asset params.
#    [DLS11] Dimitroff, Lorenz, Szimayer (2011). Parsimonious multi-asset.
#    [L08]   Lord, Koekkoek, Van Dijk (2010). Full-truncation Euler.
#    [G06]   Gatheral (2006). Milstein CIR correction.

KAPPA_DEFAULT = 5.0
XI_DEFAULT    = 0.7
RHO_DEFAULT   = -0.7

def calibrate_heston(train_returns_df, TICKERS,
                     kappa=KAPPA_DEFAULT, xi=XI_DEFAULT, rho=RHO_DEFAULT):
    """Simplified: only θ, μ from data; κ, ξ, ρ fixed (standard equity values)."""
    R = train_returns_df.values if hasattr(train_returns_df, 'values') else np.asarray(train_returns_df)
    n_days, d_ = R.shape
    mu_arr, theta_arr = np.zeros(d_), np.zeros(d_)

    print(f"\n  ┌─ HESTON CALIBRATION [B19 + DLS11] ────────────────────")
    print(f"  │  Fixed: κ={kappa}  ξ={xi}  ρ={rho}")
    for k in range(d_):
        r_k = R[:, k]
        sigma_ann    = r_k.std() * np.sqrt(252)
        theta_arr[k] = sigma_ann ** 2
        mu_arr[k]    = r_k.mean() * 252.0 + theta_arr[k] / 2.0
        feller = 2 * kappa * theta_arr[k]
        f_ok = "✓" if feller > xi ** 2 else "✗"
        print(f"  │  {TICKERS[k]:>5s}: σ={sigma_ann:.4f}  θ={theta_arr[k]:.4f}  "
              f"μ={mu_arr[k]:+.4f}  Feller:{f_ok}")

    rho_matrix = np.corrcoef(R.T)
    eigvals = np.linalg.eigvalsh(rho_matrix)
    if eigvals.min() < 0:
        rho_matrix += (-eigvals.min() + 1e-8) * np.eye(d_)
        rho_matrix /= np.sqrt(np.outer(np.diag(rho_matrix), np.diag(rho_matrix)))
    L_chol = np.linalg.cholesky(rho_matrix)
    print(f"  └───────────────────────────────────────────────────────")

    return {'mu': mu_arr, 'kappa': np.full(d_, kappa),
            'theta': theta_arr, 'xi': np.full(d_, xi),
            'rho': np.full(d_, rho), 'v0': theta_arr.copy(),
            'rho_matrix': rho_matrix, 'L_chol': L_chol,
            'tickers': list(TICKERS)}


def generate_heston(heston_params, S0, n_paths=20_000, n_steps=252,
                    n_substeps=4, use_milstein=True, seed=42,
                    drive_folder=DRIVE_FOLDER):
    """Full Truncation Euler + Milstein CIR correction."""
    cache_path = os.path.join(drive_folder, 'heston_paths.npz')
    if os.path.exists(cache_path):
        try:
            data = np.load(cache_path)
            ret, S_n = data['returns'], data['S_norm']
            if ret.shape == (n_paths, n_steps, ret.shape[2]) and np.isfinite(ret).all():
                print(f"  ⏩ Heston loaded from {cache_path}")
                return {'returns': ret, 'S_norm': S_n,
                        'variance': data.get('variance', None)}
        except Exception as e:
            print(f"  ⚠️ Cache fail: {e}")

    mu, kappa, theta = heston_params['mu'], heston_params['kappa'], heston_params['theta']
    xi, rho, v0      = heston_params['xi'], heston_params['rho'], heston_params['v0']
    L_chol           = heston_params['L_chol']
    d_               = len(mu)
    delta            = (1.0 / 252.0) / n_substeps

    print(f"\n  Generating {n_paths:,} Heston paths, substeps={n_substeps}...")
    rng = np.random.default_rng(seed)
    t0 = time.time()

    sqrt_delta  = np.sqrt(delta)
    sqrt_1_rho2 = np.sqrt(1.0 - rho ** 2)
    mil_c = (xi ** 2 / 4.0) * delta if use_milstein else None

    log_S    = np.zeros((n_paths, n_steps + 1, d_))
    v_store  = np.zeros((n_paths, n_steps + 1, d_))
    v_store[:, 0, :] = v0[None, :]
    v_cur    = v_store[:, 0, :].copy()
    logS_cur = log_S[:, 0, :].copy()

    for t in range(n_steps):
        for s in range(n_substeps):
            v_pos   = np.maximum(v_cur, 0.0)
            sqrt_vp = np.sqrt(v_pos)

            Z_a = rng.standard_normal((n_paths, d_))
            Z_i = rng.standard_normal((n_paths, d_))
            eps_S = Z_a @ L_chol.T
            eps_v = rho[None, :] * eps_S + sqrt_1_rho2[None, :] * Z_i

            v_new = (v_cur + kappa[None, :] * (theta[None, :] - v_pos) * delta
                     + xi[None, :] * sqrt_vp * sqrt_delta * eps_v)
            if use_milstein:
                v_new += mil_c[None, :] * (eps_v ** 2 - 1.0)

            logS_cur += (mu[None, :] - v_pos / 2.0) * delta + sqrt_vp * sqrt_delta * eps_S
            v_cur = v_new

        v_store[:, t + 1, :] = v_cur
        log_S[:,   t + 1, :] = logS_cur

        if (t + 1) % 50 == 0 or t == n_steps - 1:
            print(f"    Day {t+1}/{n_steps}  {time.time()-t0:.1f}s", flush=True)

    returns = np.diff(log_S, axis=1)
    S_norm  = np.exp(log_S)
    if (~np.isfinite(returns)).any():
        returns = np.nan_to_num(returns, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"\n  Done in {time.time()-t0:.1f}s")
    ret_flat = returns.reshape(-1, d_)
    for k in range(d_):
        r_k = ret_flat[:, k]
        print(f"    {heston_params['tickers'][k]}: "
              f"vol={r_k.std()*np.sqrt(252):.4f} (target {np.sqrt(theta[k]):.4f})  "
              f"kurt={stats.kurtosis(r_k)+3:.2f}")
    corr_mae = np.abs(np.corrcoef(ret_flat.T) - heston_params['rho_matrix'])[
        np.triu_indices(d_, k=1)].mean()
    print(f"    Corr-MAE: {corr_mae:.4f}")

    np.savez_compressed(cache_path, returns=returns, S_norm=S_norm,
                        variance=v_store, params_kappa=kappa, params_theta=theta,
                        params_xi=xi, params_rho=rho)
    print(f"  💾 {cache_path} ({os.path.getsize(cache_path)/1e6:.1f} MB)")

    return {'returns': returns, 'S_norm': S_norm, 'variance': v_store}

print("  ✅ GBM + Heston helper functions defined")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 7: GBM — Calibrate + Generate + Split
# ═════════════════════════════════════════════════════════════════

print("=" * 70)
print("GBM CALIBRATION & GENERATION")
print("=" * 70)

gbm_params  = calibrate_gbm(train_returns, TICKERS)
gbm_results = generate_gbm(gbm_params, S0, N_WINDOW, d, DELTA_T, DEVICE,
                           M_SIMU=M_SIMU, seed=42, drive_folder=DRIVE_FOLDER)

# ── Train/val/test split ──
S_norm = gbm_results['S_gbm_norm']
rng    = np.random.RandomState(42)
idx    = rng.permutation(S_norm.shape[0])
gbm_train = S_norm[idx[:N_TRAIN]]
gbm_val   = S_norm[idx[N_TRAIN:N_TRAIN + N_VAL]]
gbm_test  = S_norm[idx[N_TRAIN + N_VAL:]]
print(f"\n  Split: train {gbm_train.shape}  val {gbm_val.shape}  test {gbm_test.shape}")

_p = os.path.join(DRIVE_FOLDER, 'gbm_deep_hedging.npz')
np.savez_compressed(_p,
    S_train=gbm_train, S_val=gbm_val, S_test=gbm_test,
    S_norm=S_norm, returns=gbm_results['X_gbm'], S0=S0,
    tickers=np.array(TICKERS),
    drift_annual=gbm_params['drift_annual'],
    sigma_annual=gbm_params['sigma_annual'],
    rho_matrix=gbm_params['rho_matrix'])
print(f"  💾 {_p} ({os.path.getsize(_p)/1e6:.1f} MB)")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 8: HESTON — Calibrate + Generate + Split
# ═════════════════════════════════════════════════════════════════

print("=" * 70)
print("HESTON CALIBRATION & GENERATION")
print("=" * 70)

heston_params  = calibrate_heston(train_returns, TICKERS)
heston_results = generate_heston(heston_params, S0,
                                 n_paths=M_SIMU, n_steps=N_WINDOW,
                                 n_substeps=4, seed=42,
                                 drive_folder=DRIVE_FOLDER)

# ── Split ──
S_norm = heston_results['S_norm']
rng    = np.random.RandomState(42)
idx    = rng.permutation(S_norm.shape[0])
heston_train = S_norm[idx[:N_TRAIN]]
heston_val   = S_norm[idx[N_TRAIN:N_TRAIN + N_VAL]]
heston_test  = S_norm[idx[N_TRAIN + N_VAL:]]
print(f"\n  Split: train {heston_train.shape}  val {heston_val.shape}  test {heston_test.shape}")

_p = os.path.join(DRIVE_FOLDER, 'heston_deep_hedging.npz')
np.savez_compressed(_p,
    S_train=heston_train, S_val=heston_val, S_test=heston_test,
    S_norm=S_norm, returns=heston_results['returns'], S0=S0,
    tickers=np.array(TICKERS), variance=heston_results['variance'])
print(f"  💾 {_p} ({os.path.getsize(_p)/1e6:.1f} MB)")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 9: SBTS — Bandwidth Search + Generate (Config A only)
# ═════════════════════════════════════════════════════════════════

print("=" * 70)
print("SBTS — BANDWIDTH SELECTION & PATH GENERATION")
print("=" * 70)

# ── Step 1: Build reference trajectories ──
T_train = train_returns_values.shape[0]
M_ref   = T_train - N_WINDOW

X_ref = np.zeros((M_ref, N_WINDOW + 1, d))
for i in range(M_ref):
    X_ref[i, 1:, :] = train_returns_values[i : i + N_WINDOW]

print(f"  Reference paths: {X_ref.shape} (M={M_ref}, N={N_WINDOW}, d={d})")
print(f"  Training window: {train_returns.index[0].date()} – "
      f"{train_returns.index[-1].date()}")

# ── Step 2: Bandwidth selection ([A25] eq. 5) ──
bw_cache_path = os.path.join(DRIVE_FOLDER, 'bandwidth_cache.npz')
bw_loaded = False

if os.path.exists(bw_cache_path):
    try:
        c = np.load(bw_cache_path, allow_pickle=True)
        h_best = float(c['h_best'])
        k_best = int(c['k_best'])
        bw_meta = c['bw_meta'].item()
        bw_loaded = True
        print(f"\n  ⏩  Bandwidth cache loaded: h*={h_best:.4f}  k*={k_best}")
    except Exception as e:
        print(f"  ⚠️  Cache load failed: {e} → recomputing")

if not bw_loaded:
    h_best, k_best, bw_meta = select_bandwidth_and_markov_k(
        X_ref=X_ref, d=d, TICKERS=TICKERS, deltati=DELTA_T, device=DEVICE,
        Q=100, L=50, test_fraction=0.2, seed=42, verbose=True)
    np.savez_compressed(bw_cache_path,
        h_best=np.array(h_best), k_best=np.array(k_best),
        bw_meta=np.array(bw_meta))
    print(f"  💾 Bandwidth cache saved: {bw_cache_path}")

# ── Step 3: Generate paths using best (h, k) ──
print(f"\n" + "=" * 70)
print(f"SBTS PATH GENERATION — h={h_best:.4f}  K={k_best}")
print("=" * 70)

sbts_cache = os.path.join(DRIVE_FOLDER, 'sbts_paths.npz')
sbts_loaded = False

if os.path.exists(sbts_cache):
    try:
        data = np.load(sbts_cache)
        X_sbts = data['returns']
        S_norm_sbts = data['S_norm']
        if X_sbts.shape == (M_SIMU, N_WINDOW, d) and np.isfinite(X_sbts).all():
            sbts_loaded = True
            print(f"  ⏩ SBTS paths loaded from {sbts_cache}")
    except Exception as e:
        print(f"  ⚠️ Cache load failed: {e}")

if not sbts_loaded:
    t0 = time.time()
    X_sbts = simulate_sbts(
        N=N_WINDOW, M=M_ref, d=d, K=k_best,
        X=X_ref, N_pi=N_PI, h=h_best, deltati=DELTA_T,
        M_simu=M_SIMU, device=DEVICE, batch_size=BATCH_SIZE, seed=42,
        config_name='A', drive_folder=DRIVE_FOLDER)
    print(f"\n  Done in {time.time() - t0:.1f}s")

    S_norm_sbts = returns_to_S_norm(X_sbts, M_SIMU, N_WINDOW, d)
    np.savez_compressed(sbts_cache,
        returns=X_sbts, S_norm=S_norm_sbts, S0=S0,
        tickers=np.array(TICKERS),
        h_best=np.array(h_best), k_best=np.array(k_best))
    print(f"  💾 {sbts_cache} ({os.path.getsize(sbts_cache)/1e6:.1f} MB)")
    cleanup_temp_batches('A', DRIVE_FOLDER)

# ── Step 4: Split train/val/test ──
rng = np.random.RandomState(42)
idx = rng.permutation(X_sbts.shape[0])
sbts_train = S_norm_sbts[idx[:N_TRAIN]]
sbts_val   = S_norm_sbts[idx[N_TRAIN:N_TRAIN + N_VAL]]
sbts_test  = S_norm_sbts[idx[N_TRAIN + N_VAL:]]

print(f"\n  Split: train {sbts_train.shape}  val {sbts_val.shape}  test {sbts_test.shape}")

_p = os.path.join(DRIVE_FOLDER, 'sbts_deep_hedging.npz')
np.savez_compressed(_p,
    S_train=sbts_train, S_val=sbts_val, S_test=sbts_test,
    S_norm=S_norm_sbts, returns=X_sbts, S0=S0,
    tickers=np.array(TICKERS),
    h_best=np.array(h_best), k_best=np.array(k_best))
print(f"  💾 {_p} ({os.path.getsize(_p)/1e6:.1f} MB)")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 10: HISTORICAL TEST PATHS (4 periods)
# ═════════════════════════════════════════════════════════════════
#  For each start date in a period, build a 252-day forward window
#  of normalised prices (S_t / S_0 = 1).
#  Windows starting late in a period extend into subsequent periods —
#  this is by design (option must be hedged over full maturity).
# ═════════════════════════════════════════════════════════════════

print("=" * 70)
print("BUILDING HISTORICAL TEST PATHS — 4 periods")
print("=" * 70)

historical_tests = {}

for period_name, (start, end) in TEST_PERIODS.items():
    mask = (full_returns.index >= start) & (full_returns.index <= end)
    start_dates = full_returns.index[mask]

    paths_list = []
    for sd in start_dates:
        start_idx = full_returns.index.get_loc(sd)
        end_idx   = start_idx + N_WINDOW
        if end_idx > len(full_returns):
            break
        window_returns = full_returns.values[start_idx:end_idx]
        cum_returns = np.cumsum(window_returns, axis=0)
        S_path = np.ones((N_WINDOW + 1, d))
        S_path[1:, :] = np.exp(cum_returns)
        paths_list.append(S_path)

    if not paths_list:
        print(f"  {period_name}: NO paths (insufficient data)")
        continue

    S_hist = np.stack(paths_list, axis=0)
    historical_tests[period_name] = S_hist
    print(f"  {period_name:>20s}: {S_hist.shape[0]:>4d} paths  "
          f"[{start} → {end}]")

all_hist_paths = np.concatenate(list(historical_tests.values()), axis=0)
print(f"\n  Total historical paths: {all_hist_paths.shape[0]}")

# ── Save ──
hp = os.path.join(DRIVE_FOLDER, 'historical_test_paths.npz')
save_dict = {f'S_norm_{k}': v for k, v in historical_tests.items()}
save_dict.update({
    'S_norm_all':   all_hist_paths,
    'S0':           S0,
    'tickers':      np.array(TICKERS),
    'period_names': np.array(list(historical_tests.keys())),
    'period_sizes': np.array([v.shape[0] for v in historical_tests.values()]),
})
np.savez_compressed(hp, **save_dict)
print(f"  💾 {hp} ({os.path.getsize(hp)/1e6:.1f} MB)")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 11: THREE-WAY DISTRIBUTIONAL COMPARISON
# ═════════════════════════════════════════════════════════════════

print("=" * 70)
print("DISTRIBUTIONAL COMPARISON — Historical vs GBM / Heston / SBTS")
print("=" * 70)

triu = np.triu_indices(d, k=1)

# Reshape returns for comparison
hist_flat   = train_returns_values
gbm_flat    = gbm_results['X_gbm'].reshape(-1, d)
heston_flat = heston_results['returns'].reshape(-1, d)
sbts_flat   = X_sbts.reshape(-1, d)

flats = {'Hist': hist_flat, 'GBM': gbm_flat,
         'Heston': heston_flat, 'SBTS': sbts_flat}

def compute_stats(flat, hist=None):
    out = {
        'std_ann':  flat.std(axis=0) * np.sqrt(252),
        'kurtosis': np.array([stats.kurtosis(flat[:, k]) + 3 for k in range(d)]),
        'skewness': np.array([stats.skew(flat[:, k]) for k in range(d)]),
        'corr':     np.corrcoef(flat.T),
    }
    if hist is not None:
        out['ks'] = np.array([
            stats.ks_2samp(hist[:, k], flat[:, k]).statistic
            for k in range(d)])
    else:
        out['ks'] = np.zeros(d)

    def td_lower(x, y, q=0.05):
        n = len(x)
        u = stats.rankdata(x) / (n + 1)
        v = stats.rankdata(y) / (n + 1)
        return np.mean((u <= q) & (v <= q)) / q

    out['td_L'] = np.array([td_lower(flat[:, i], flat[:, j])
                            for i, j in zip(*triu)])
    return out

diags = {}
diags['Hist'] = compute_stats(hist_flat, hist=None)
for key in ['GBM', 'Heston', 'SBTS']:
    diags[key] = compute_stats(flats[key], hist=hist_flat)

# ── Table 1: Three-way summary ──
print("\n" + "─" * 70)
print(f"{'Metric':<22s} {'Hist':>10s} {'GBM':>10s} {'Heston':>10s} {'SBTS':>10s}")
print("─" * 70)

kurt = {k: diags[k]['kurtosis'].mean() for k in diags}
print(f"{'Avg kurtosis':<22s} {kurt['Hist']:>10.2f} {kurt['GBM']:>10.2f} "
      f"{kurt['Heston']:>10.2f} {kurt['SBTS']:>10.2f}")

for k in ['GBM', 'Heston', 'SBTS']:
    gap = abs(kurt[k] - kurt['Hist'])
    print(f"  |Δkurtosis| {k:<10s} {'':>10s} "
          f"{gap:>10.2f}" if k=='GBM' else
          f"  |Δkurtosis| {k:<10s}{'':>20s}{gap:>10.2f}")

# Nicely formatted gaps
print()
print(f"{'|Δkurtosis|':<22s} {'—':>10s} "
      f"{abs(kurt['GBM']-kurt['Hist']):>10.2f} "
      f"{abs(kurt['Heston']-kurt['Hist']):>10.2f} "
      f"{abs(kurt['SBTS']-kurt['Hist']):>10.2f}")

corr_mae = {k: np.abs(diags[k]['corr'] - diags['Hist']['corr'])[triu].mean()
            for k in ['GBM', 'Heston', 'SBTS']}
print(f"{'Corr-MAE':<22s} {'—':>10s} "
      f"{corr_mae['GBM']:>10.4f} {corr_mae['Heston']:>10.4f} "
      f"{corr_mae['SBTS']:>10.4f}")

ks_mean = {k: diags[k]['ks'].mean() for k in ['GBM', 'Heston', 'SBTS']}
print(f"{'Avg KS statistic':<22s} {'—':>10s} "
      f"{ks_mean['GBM']:>10.4f} {ks_mean['Heston']:>10.4f} "
      f"{ks_mean['SBTS']:>10.4f}")

td_L_mean = {k: diags[k]['td_L'].mean() for k in diags}
print(f"{'Lower tail dep (5%)':<22s} {td_L_mean['Hist']:>10.3f} "
      f"{td_L_mean['GBM']:>10.3f} {td_L_mean['Heston']:>10.3f} "
      f"{td_L_mean['SBTS']:>10.3f}")

td_gap = {k: abs(td_L_mean[k] - td_L_mean['Hist'])
          for k in ['GBM', 'Heston', 'SBTS']}
print(f"{'|Δλ_L|':<22s} {'—':>10s} "
      f"{td_gap['GBM']:>10.3f} {td_gap['Heston']:>10.3f} "
      f"{td_gap['SBTS']:>10.3f}")
print("─" * 70)

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 12: FIGURE 2 — Return Distributions (KDE), 1×3 panel
# ═════════════════════════════════════════════════════════════════

colors_gen = {'Hist': '#2C3E50', 'GBM': '#E74C3C',
              'Heston': '#27AE60', 'SBTS': '#2980B9'}

fig, axes = plt.subplots(1, d, figsize=(11, 3.5), dpi=150, sharey=True)

for k in range(d):
    ax = axes[k]
    for key in ['Hist', 'GBM', 'Heston', 'SBTS']:
        data = flats[key][:, k] * 100   # as %
        lo, hi = np.percentile(data, [0.5, 99.5])
        clip = data[(data >= lo) & (data <= hi)]
        kde = stats.gaussian_kde(clip)
        x = np.linspace(-8, 8, 400)
        ls = '-' if key == 'Hist' else '--'
        lw = 1.7 if key == 'Hist' else 1.1
        ax.plot(x, kde(x), color=colors_gen[key], ls=ls, lw=lw, label=key)
    ax.set_xlabel('Daily return (%)')
    if k == 0:
        ax.set_ylabel('Density')
        ax.legend(frameon=False, loc='upper right')
    ax.set_title(TICKERS[k], fontsize=11)
    ax.set_xlim(-6, 6)

plt.tight_layout()
plt.savefig(os.path.join(FIG_FOLDER, 'fig2_return_distributions.pdf'),
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIG_FOLDER, 'fig2_return_distributions.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print(f"  💾 fig2_return_distributions.pdf/png")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 13: FIGURE 3 — Lower Tail Dependence (bars) + Corr Heatmap
# ═════════════════════════════════════════════════════════════════
#  Two-panel:
#   (a) Lower tail dep λ_L at q=5%, per pair — which generator
#       preserves joint tail behaviour?
#   (b) Correlation matrix heatmaps (4 × d×d) side by side
# ═════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=150,
                         gridspec_kw={'width_ratios': [1, 1.3]})

# ── (a) Lower tail dependence bar chart ──
ax = axes[0]
pair_labels = [f'{TICKERS[i]}-{TICKERS[j]}' for i, j in zip(*triu)]
x = np.arange(len(pair_labels))
w = 0.2

for idx, key in enumerate(['Hist', 'GBM', 'Heston', 'SBTS']):
    vals = diags[key]['td_L']
    ax.bar(x + (idx - 1.5) * w, vals, w,
           color=colors_gen[key], label=key,
           alpha=0.9 if key == 'Hist' else 0.75,
           edgecolor='black' if key == 'Hist' else None, lw=0.5)

ax.set_xticks(x)
ax.set_xticklabels(pair_labels, fontsize=9)
ax.set_ylabel(r'Lower tail dependence $\hat{\lambda}_L$ (q=5%)')
ax.set_title('(a) Joint Tail Behaviour', loc='left', fontweight='bold', fontsize=11)
ax.legend(frameon=False, fontsize=9, ncol=4, loc='upper center',
          bbox_to_anchor=(0.5, 1.15))
ax.set_ylim(0, max(diags['Hist']['td_L']) * 1.35)

# ── (b) Correlation heatmaps in a grid ──
ax = axes[1]
ax.axis('off')

# Draw 4 compact heatmaps using imshow on sub-axes
from matplotlib.gridspec import GridSpecFromSubplotSpec
gs = GridSpecFromSubplotSpec(1, 4, subplot_spec=ax.get_subplotspec(),
                             wspace=0.3)

for i, key in enumerate(['Hist', 'GBM', 'Heston', 'SBTS']):
    sub_ax = fig.add_subplot(gs[0, i])
    M = diags[key]['corr']
    im = sub_ax.imshow(M, cmap='RdYlBu_r', vmin=-0.2, vmax=1.0, aspect='equal')
    sub_ax.set_title(key, fontsize=10, fontweight='bold')
    sub_ax.set_xticks(range(d))
    sub_ax.set_yticks(range(d))
    sub_ax.set_xticklabels(TICKERS, fontsize=7, rotation=45)
    sub_ax.set_yticklabels(TICKERS, fontsize=7)
    # overlay values
    for ii in range(d):
        for jj in range(d):
            color = 'white' if abs(M[ii, jj]) > 0.6 else 'black'
            sub_ax.text(jj, ii, f'{M[ii,jj]:.2f}',
                        ha='center', va='center', fontsize=7, color=color)

fig.text(0.52, 0.93, '(b) Correlation Matrix', fontweight='bold',
         fontsize=11, ha='left')

plt.tight_layout()
plt.savefig(os.path.join(FIG_FOLDER, 'fig3_tail_and_correlation.pdf'),
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIG_FOLDER, 'fig3_tail_and_correlation.png'),
            dpi=300, bbox_inches='tight')
plt.show()
print(f"  💾 fig3_tail_and_correlation.pdf/png")

In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 14: PIPELINE SUMMARY
# ═════════════════════════════════════════════════════════════════

print("=" * 70)
print("DATA PIPELINE SUMMARY")
print("=" * 70)

files = [
    'gbm_deep_hedging.npz',
    'heston_deep_hedging.npz',
    'sbts_deep_hedging.npz',
    'historical_test_paths.npz',
    'bandwidth_cache.npz',
]

print(f"\n  Drive folder: {DRIVE_FOLDER}")
print(f"  {'File':<38s} {'Size':>10s} {'Status':>10s}")
print(f"  {'-' * 60}")
for f in files:
    fp = os.path.join(DRIVE_FOLDER, f)
    if os.path.exists(fp):
        sz = f"{os.path.getsize(fp)/1e6:.1f} MB"
        print(f"  {f:<38s} {sz:>10s} {'✅':>10s}")
    else:
        print(f"  {f:<38s} {'---':>10s} {'❌ MISSING':>10s}")

print(f"\n  Figures in {FIG_FOLDER}:")
for f in sorted(os.listdir(FIG_FOLDER)):
    fp = os.path.join(FIG_FOLDER, f)
    print(f"    {f}  ({os.path.getsize(fp)/1e3:.0f} KB)")

print(f"\n  SBTS hyperparameters (selected by held-out MSE):")
print(f"    h* = {h_best:.4f}")
print(f"    k* = {k_best}")

print(f"\n  Paths per generator: {M_SIMU:,}")
print(f"  Splits: train {N_TRAIN} / val {N_VAL} / test {N_TEST}")

print(f"\n  Historical test periods:")
for name, (s, e) in TEST_PERIODS.items():
    n = historical_tests.get(name, np.array([])).shape[0] if name in historical_tests else 0
    print(f"    {name:<20s} {s} → {e}  ({n} paths)")

print(f"\n  ✅ Ready for deep hedging training")
print(f"     Next: article_2_gbm.ipynb, article_3_heston.ipynb, article_4_sbts.ipynb")